In [ ]:
#!/user/bin/env python3

! pip install biopython
! pip install -q condacolab
import condacolab
condacolab.install()
! conda install -c bioconda seqkit


In [31]:
import requests
import shutil
import subprocess
import sys
import builtins
from Bio import SeqIO

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        headers = {"accept": "application/json"}
        url = f"https://rest.uniprot.org/uniprotkb/{accession}"
        return requests.get(url, headers=headers)

    def _get_ensembl(self, id):
        headers = {"accept": "application/json"}
        url = f"https://rest.ensembl.org/lookup/id/{id}"
        return requests.get(url, headers=headers)

    @staticmethod
    def _uniprot_parse_response(resp):
        if not resp.ok:
            resp.raise_for_status()
        data = resp.json()
        return {
            "species": data.get("organism", {}).get("scientificName", ""),
            "geneInfo": data.get("genes", []),
            "sequence": data.get("sequence", {}),
            "type": "protein",
        }

    @staticmethod
    def _ensembl_parse_response(resp):
        if not resp.ok:
            resp.raise_for_status()
        data = resp.json()
        attrs = ["object_type", "assembly_name", "species", "db_type",
                 "biotype", "display_name", "description",
                 "canonical_transcript", "source"]
        return {a: data[a] for a in attrs if a in data}

    def _access_database(self, id, database, seq_description, seq_sequence):
        output = {"seq_description": seq_description, "seq_sequence": seq_sequence}
        try:
            if database.lower() == "uniprot":
                resp = self._get_uniprot(id)
                if resp.ok:
                    output.update(self._uniprot_parse_response(resp))
                else:
                    output["error"] = f"Uniprot request failed: {resp.status_code}"
            elif database.lower() == "ensembl":
                resp = self._get_ensembl(id)
                if resp.ok:
                    output.update(self._ensembl_parse_response(resp))
                else:
                    output["error"] = f"ensembl request failed: {resp.status_code}"
        except Exception as e:
            output["error"] = f"Exception during request: {e}"
        return output

    def seqkit_stats(self):
        try:
            proc = subprocess.run(
                ("seqkit", "stats", self.filename, "-a"),
                capture_output=True, text=True, check=False
            )
            if proc.stderr:
                return {"ERROR": proc.stderr}
            lines = proc.stdout.strip().split('\n')
            if len(lines) < 2:
                return {"ERROR": "bad output from seqkit"}
            names = lines[0].split()[1:]
            values = lines[1].split()[1:]
            return dict(zip(names, values))
        except Exception as e:
            return {"ERROR": f"exception running seqkit: {e}"}

    def _biopython_stats(self):
        try:
            records = list(SeqIO.parse(self.filename, "fasta"))
            if not records:
                return {"ERROR": "No sequences found"}
            lengths = [len(rec) for rec in records]
            return {
                "num_seqs": builtins.str(len(records)),
                "min_len": builtins.str(min(lengths)),
                "max_len": builtins.str(max(lengths)),
                "avg_len": f"{sum(lengths)/len(lengths):.2f}",
                "total_len": builtins.str(sum(lengths)),
                "format": "fasta"
            }
        except Exception as e:
            return {"ERROR": f"Biopython stats failed: {e}"}

    def biopython_parser(self, seqkit_result):
        if "ERROR" in seqkit_result:
            return seqkit_result

        ext = seqkit_result.get('format', 'fasta').lower()
        output = {}
        try:
            sequences = SeqIO.parse(self.filename, ext)
        except Exception as e:
            return {"ERROR": f"Failed to parse file: {e}"}

        for seq in sequences:
            header = seq.description
            if header.upper().startswith('ENS'):
                database = 'ensembl'
                identifier = header.split()[0]
            else:
                database = 'uniprot'
                parts = header.split('|')
                identifier = parts[1] if len(parts) >= 2 else header.split()[0]

            seq_str = builtins.str(seq.seq)
            info = self._access_database(identifier, database, header, seq_str)
            output[seq.id] = {
                "header": header,
                "sequence": seq_str,
                "database_info": info
            }
        return output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print("\t" * indent + builtins.str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print("\t" * (indent + 1) + builtins.str(value))

In [32]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print(stats)

{'format': 'FASTA', 'type': 'Protein', 'num_seqs': '2', 'sum_len': '456', 'min_len': '29', 'avg_len': '228', 'max_len': '427', 'Q1': '29', 'Q2': '228', 'Q3': '427', 'sum_gap': '0', 'N50': '427', 'N50_num': '1', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '0', 'sum_n': '0'}


In [33]:
biopython = parser.biopython_parser(stats)

parser.show_output(biopython)

sp|P11473|VDR_HUMAN
	header
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
	database_info
		seq_description
			sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
		seq_sequence
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLI

In [34]:
parser1 = MyFastaParser('ensembl_download_1.fasta')
stats1 = parser1.seqkit_stats()
print(stats1)

{'format': 'FASTA', 'type': 'DNA', 'num_seqs': '6', 'sum_len': '86', 'min_len': '9', 'avg_len': '14.3', 'max_len': '23', 'Q1': '10', 'Q2': '13.5', 'Q3': '17', 'sum_gap': '0', 'N50': '16', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '45.35', 'sum_n': '0'}


In [35]:
biopython1 = parser1.biopython_parser(stats1)

parser1.show_output(biopython1)

ENSMUST00000196221.2
	header
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
	database_info
		seq_description
			ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
		seq_sequence
			ATGGCATAT
		error
			Ensembl request failed: 400
ENSMUST00000177564.2
	header
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
	sequence
		ATCGGAGGGATACGAG
	database_info
		seq_description
			ENSMUST000001775

In [39]:
parser2 = MyFastaParser('uniport_download.fasta')
stats2 = parser2.seqkit_stats()
print(stats2)

{'ERROR': '\x1b[ERRO]\x1b stat uniport_download.fasta: no such file or directory\n'}


In [40]:
biopython2 = parser2.biopython_parser(stats2)

parser2.show_output(biopython)

ERROR
	[ERRO] stat uniport_download.fasta: no such file or directory

